## 1. Basic Tasks

**1. Use CTAS with read_files() to ingest a CSV file into a managed Delta table.**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev.bronze.sales_new AS 
SELECT *
FROM read_files(
    "/Volumes/dev/bronze/raw/sales.csv",
    format => "csv",
    header => true,
    inferSchema => true 
)

**2. Ingest a nested JSON file, extracting at least 2 nested fields into top-level columns.**

In [0]:
df = spark.read.option("multiline", "true").json("/Volumes/dev/bronze/raw/orders.json")

In [0]:
from pyspark.sql.functions import *
df_flattend = df.select(
    col("customer_id"),
    col("name"),
    col("address.city").alias("city"),
    col("address.state").alias("state"),
    explode(col("orders")).alias("single_order")
)

In [0]:
df_flattend.write.mode("overwrite").saveAsTable("dev.bronze.orders_json")

**3. Run DESCRIBE, DESCRIBE EXTENDED, and DESCRIBE DETAIL on your new table and note what unique
information each one gives you.**

In [0]:
%sql
DESC dev.bronze.sales_new

In [0]:
%sql
DESC EXTENDED dev.bronze.sales_new

In [0]:
%sql
DESC DETAIL dev.bronze.sales_new

* DESCRIBE shows only column names, column types and column comments of the table.
* DESCRIBE EXTENDED shows table details and also the metadata. It shows catalog, schema and table name and also it shows type, location, provider, owner and table properties.
* DESCRIBE DETAIL shows format, numFiles, sizeInBytes, minReaderVersion/ minWriterVersion, partitionColumns, cratedAt, lastModified. It is a singular column showing all the details.

## 2. Intermediate Tasks

**4. Add _metadata.file_name and _metadata.file_path to your ingestion query and use them to prove
which source file each row came from.**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev.bronze.sales_with_metadata AS
SELECT *, 
    _metadata.file_name AS file_name,
    _metadata.file_path AS file_path,
    CURRENT_TIMESTAMP() AS ingestion_timestamp
FROM read_files(
    "/Volumes/dev/bronze/raw/sales.csv",
    format => "csv",
    header => true,
    inferSchema => true
);

**5. Create an Iceberg table from the same source data and compare its DESCRIBE DETAIL output (format,
location) to the Delta version.**

In [0]:
%sql
CREATE TABLE dev.bronze.slaes_new_iceberg 
USING iceberg AS
SELECT *
FROM read_files (
    "/Volumes/dev/bronze/raw/sales.csv",
    format => "csv",
    header => true,
    inferSchema => true
);

In [0]:
%sql
DESC DETAIL dev.bronze.slaes_new_iceberg;

In case of the delta the format is Delta and location column is empty as it managed by databricks internally.
IN case of the icebarg the format is iceberg and the location column shows amazon s3 storage location

**6. (Data Analyst) Write a query using the metadata columns to build a 'records per source file' audit
report — useful for verifying a vendor's daily file drop.**

In [0]:
%sql
SELECT 
    file_name AS vendor_file_name,
    file_path AS file_location,
    COUNT(*) AS total_records_loaded,
    MIN(ingestion_timestamp) AS first_record_ingested_at,
    MAX(ingestion_timestamp) AS last_record_ingested_at
FROM dev.bronze.sales_with_metadata
GROUP BY file_name, file_path
ORDER BY first_record_ingested_at DESC;

## 3. Advanced Tasks

**7. Land multiple CSV files with slightly different formats (e.g., an extra column, a different delimiter)
and document how your read_files() options need to change for each, plus how you'd detect a
mismatch before it silently breaks downstream reports.**

i. Handling Extra or Missing Columns

In [0]:
%sql
CREATE OR REPLACE TABLE dev.bronze.sales_read 
AS SELECT *
FROM read_files(
    "/Volumes/dev/bronze/raw/sales_read/",
    header => True,
    schema => 'order_id INT, customer_id INT, transaction_id INT, product_id INT, quantity INT, discount_amount DOUBLE, total_amount DOUBLE, order_date DATE',
    rescuedDataColumn => "rescued_data"
);

In [0]:
%sql
select * from dev.bronze.sales_read 

To read multiple csv files with slightly different format we can use just `read_files`, this `read_files` reads all the schema of those files and, as I have two files one with an extra column payment_method it the table that was created wil have the extra column and for the data of the first file the column value of payment_method will be null, and in case of different delimiters we can add 
`sep => ',' or '|'` inside the `read_files`, so for that we can also handle that.

But other than the extra column read_files also create another column `_rescued_data` which is very usefull for checking the mismatch that was there in the downstream reports. 

the `_rescuse_data` column helps rescues data in case any problematic data appears then that engine put that data in the `_rescuse_data` column.
The `_rescuse_data` data column rescues data in case of extra column, different data type and the column name miss match, so instead of droping the data entierly and putting null value, the engine puts the null value in the original column and put the orignal bad data in the `_rescuse_data` column.

In [0]:
%sql
desc dev.bronze.sales_read

**8. Write a decision memo: when should Cyntexa choose Iceberg (or Delta UniForm) over native Delta
for a given table, considering downstream tools like Snowflake or Trino?**

Depending on the context we can choose what works best for Cyntexa. So below I have explained when I would use what.

**1. When to use Native Delta**
- i. When the work is 100% DataBricks native: We will choose this if our work is entirely in the Databricks ecosystem.
- ii. Streaming data: When we do Structured Streaming with micro-batch processing then in that case Native Delta works best.
- iii. Databricks features: When we want DataBricks features like Governance using Unity Catalog, Liquid Clustering, etc.

**2. When to use Apache Iceberg**
- i. Working on SnowFlake: If we are working specifically on SnowFlake then we would use Iceberg, because snowflake provides better integration and also reads really fast.
- ii. Trino-focused: If you use trino to search across many different systems then we can use to keep everything organized.
- iii. Advanced Partitioning Needs: When the dataset requires partition evolution like when we need to change partitioning, from lets say daily to hourly, also when we need hidden partitioning.

**3. When to use Delta Uniform**
- i. Best of Both: Let's say if we use DataBricks to write data but Snowflake to read it then, we can use Delta Uniform.
- ii. Automatic translation: It saves the data in delta format but when you need it it can also transform the data in background so that Snowflake can read it, without duplicating the data.
- iii. Standardization without Migration: If Cyntexa already have pipelines built on Delta Lake but when we face integration difficulty with clients using Snowflake or Trino then we should use Delta Uniform.


**9. Use DESCRIBE HISTORY together with the metadata columns to trace a specific bad row back to the
exact ingestion run and source file that introduced it.**

To find or trace the specific bad rows and exact ingestion run we need the metadata columns time stamp, file path, file name and then using DESCRIBE HISTORY we can find the exact operation from the timestamp and also we can check the versions and can go back to the previous versions.

In [0]:
%sql
INSERT INTO dev.bronze.sales_with_metadata
SELECT *,
    _metadata.file_name AS file_name,
    _metadata.file_path AS file_path,
    current_timestamp() AS ingestion_timestamp
FROM read_files(
    "/Volumes/dev/bronze/raw/sales_integer_quantity.csv",
    header => "true",
    format => "csv"
)

In [0]:
%sql
SELECT * FROM dev.bronze.sales_with_metadata
WHERE quantity < 1;

In [0]:
%sql
DESC HISTORY dev.bronze.sales_with_metadata

So from the above select query I can check all the data where the quantity in less than 1, and also I can check the file_name, file_path, and ingestion_timestamp and from the DESC HISTORY I can check the exact time stamp that matches the time stamp of the column time_stamp and can check the operation that cause it and also I can check the previous versions and go back to the previous version.